# Build Drivers Dimension

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
 %run ../00-common/1.environment_config

In [0]:
%run ../00-common/4.gold_helpers

In [0]:
drivers_table = f"{catalog_name}.{silver_schema}.drivers"
ref_nationality_region_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"
# target_table
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

## Reading drivers table and gold.ref_nationality_region

In [0]:
drivers_df = (
    spark.table(drivers_table)
    .filter(F.col("batch_id") == v_batch_id)
)
ref_nationality_region_df = spark.table(ref_nationality_region_table)

## Joining the 2 tables

In [0]:
dim_drivers_df = (
    drivers_df
    .join(
        ref_nationality_region_df,
        drivers_df.nationality == ref_nationality_region_df.nationality,
        how = "left"
    )
    .select(
        drivers_df.driver_id,
        drivers_df.driver_name,
        drivers_df.date_of_birth,
        drivers_df.nationality,
        ref_nationality_region_df.region.alias("nationality_region")
    )
)

In [0]:
display(dim_drivers_df)

## Writing into the Gold Delta Table

In [0]:
dim_drivers_columns_to_update = [
    "driver_name",
    "date_of_birth",
    "nationality",
    "nationality_region"
]

write_to_gold(
    source_df=dim_drivers_df,
    target_table=target_table,
    merge_condition= "t.driver_id = s.driver_id",
    columns_to_update = dim_drivers_columns_to_update
)

In [0]:
spark.table(target_table).display()